# TradeFlow ML Training
Train a model on backtest data to predict trade outcomes.

In [ ]:
!pip install scikit-learn pandas matplotlib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import classification_report
import matplotlib.pyplot as plt


In [ ]:
df = pd.read_csv("tradeflow_training_dataset.csv")
print(f"Total trades: {len(df)}")
print(df["outcome"].value_counts())
df.head()


In [ ]:
FEATURES = [
    "patternEncoded", "direction", "patternConfidence",
    "htfBias", "htfStrength", "htfEmaSeparation",
    "rsi", "rsiZone", "macdSign", "macdMomentum",
    "atrPct", "bbPosition", "volumeRatio", "highVolume",
    "regimeEncoded", "rrRatio", "distToSupport",
    "distToResistance", "hourUTC", "dayOfWeek",
    "consecutiveLossesBefore"
]

X = df[FEATURES].fillna(0)
y = df["outcomeEncoded"].map({1: 1, 0: 0, -1: 0})  # binary: win vs not-win
y_r = df["rMultiple"]  # regression target


In [ ]:
# Walk-forward cross-validation — NEVER shuffle time series data
tscv = TimeSeriesSplit(n_splits=5)

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=4,
    class_weight="balanced",
    random_state=42
)

scores = []
for train_idx, test_idx in tscv.split(X):
    X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
    y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]
    model.fit(X_tr, y_tr)
    scores.append(model.score(X_te, y_te))

print(f"CV Accuracy: {np.mean(scores):.3f}")


In [ ]:
importance = pd.Series(
    model.feature_importances_, index=FEATURES
).sort_values(ascending=True)

plt.figure(figsize=(10, 8))
importance.plot(kind="barh")
plt.title("Feature Importance — What predicts wins?")
plt.tight_layout()
plt.show()
